In [12]:
from qiskit.quantum_info import SparsePauliOp
from qiskit.quantum_info import Operator

H_dense = Hloc.toarray()

H_op = Operator(H_dense)

H_pauli = SparsePauliOp.from_operator(H_op)

NameError: name 'Hloc' is not defined

In [ ]:
# ============================================================
# Convert scalar-field Hamiltonian to Pauli form
# and run genuine Qiskit VQE
# ============================================================

import numpy as np

from qiskit.quantum_info import (
    Operator,
    SparsePauliOp,
    Statevector
)

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA

from qiskit.primitives import StatevectorEstimator

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector


# ============================================================
# Variational circuit
# ============================================================

def variational_circuit(nq, layers):

    qc = QuantumCircuit(nq)

    nparams = 2 * nq * layers

    params = ParameterVector("θ", nparams)

    p = 0

    for l in range(layers):

        # ----------------------------------------------------
        # Single-qubit rotations
        # ----------------------------------------------------

        for q in range(nq):

            qc.ry(params[p], q)
            qc.rz(params[p+1], q)

            p += 2

        # ----------------------------------------------------
        # Entanglement layer
        # ----------------------------------------------------

        if l != layers - 1:

            # even bonds
            if l % 2 == 0:

                for q in range(0, nq-1, 2):
                    qc.cz(q, q+1)

            # odd bonds
            else:

                for q in range(1, nq-1, 2):
                    qc.cz(q, q+1)

    return qc


# ============================================================
# Dense Hamiltonian
# ============================================================

H_dense = Hloc.toarray()

# ============================================================
# Convert to Pauli operator
# ============================================================

print("Converting Hamiltonian to Pauli basis...")

H_op = Operator(H_dense)

H_pauli = SparsePauliOp.from_operator(H_op)

# remove tiny coefficients
H_pauli = H_pauli.simplify(atol=1e-10)

print("Done.")

print("Number of Pauli terms =", len(H_pauli))


# ============================================================
# Ansatz
# ============================================================

nqubits = 12
layers = 5

ansatz = variational_circuit(
    nqubits,
    layers
)

# ============================================================
# Estimator
# ============================================================

estimator = StatevectorEstimator()

# ============================================================
# Optimizer
# ============================================================

optimizer = COBYLA(
    maxiter=300
)

# ============================================================
# VQE
# ============================================================

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# Run VQE
# ============================================================

print("\nRunning VQE...\n")

result = vqe.compute_minimum_eigenvalue(
    H_pauli
)

# ============================================================
# Energy
# ============================================================

E_vqe = result.eigenvalue.real

# ============================================================
# Optimal parameters
# ============================================================

optimal_params = result.optimal_parameters

# ============================================================
# Build optimized circuit
# ============================================================

optimal_circuit = ansatz.assign_parameters(
    optimal_params
)

# ============================================================
# VQE state
# ============================================================

psi_vqe = Statevector.from_instruction(
    optimal_circuit
).data

# ============================================================
# Fidelity with exact ground state
# ============================================================

Fidelity = np.abs(
    np.vdot(psi_loc, psi_vqe)
)**2

# ============================================================
# Results
# ============================================================

print("\n================================")
print("Exact energy :", E0_loc)
print("VQE energy   :", E_vqe)
print("Energy error :", abs(E_vqe - E0_loc))
print("Fidelity     :", Fidelity)
print("Pauli terms  :", len(H_pauli))
print("================================")

In [7]:
# Convert Hamiltonian to Pauli operator
H_pauli_shifted = qubit_hamiltonian

layers = 2

ansatz = hva_circuit(
    H_pauli_shifted,
    nqubits,
    layers
)

NameError: name 'qubit_hamiltonian' is not defined

In [9]:
# ============================================================
# Hamiltonian Variational Ansatz (HVA) VQE
# ============================================================

import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector

from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import SPSA

from qiskit.primitives import StatevectorEstimator

In [10]:
print(globals().keys())

dict_keys(['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh', 'In', 'Out', 'get_ipython', 'exit', 'quit', 'open', '_', '__', '___', '__session__', '_i', '_ii', '_iii', '_i1', 'np', 'QuantumCircuit', 'ParameterVector', 'Statevector', 'VQE', 'SPSA', 'StatevectorEstimator', 'hva_circuit', 'nqubits', 'layers', '_i2', '_i3', '_i4', '_i5', '_i6', '_i7', '_i8', '_i9', '_i10'])


In [11]:
# ============================================================
# HVA Circuit
# ============================================================

def hva_circuit(H_pauli, nq, layers):

    qc = QuantumCircuit(nq)

    # --------------------------------------------------------
    # Number of Pauli terms
    # --------------------------------------------------------

    pauli_terms = list(zip(
        H_pauli.paulis,
        H_pauli.coeffs
    ))

    nterms = len(pauli_terms)

    # --------------------------------------------------------
    # Parameters
    # --------------------------------------------------------

    params = ParameterVector(
        "θ",
        nterms * layers
    )

    p = 0

    # --------------------------------------------------------
    # Initial reference state
    # --------------------------------------------------------

    for q in range(nq):
        qc.h(q)

    # ========================================================
    # HVA layers
    # ========================================================

    for l in range(layers):

        for pauli, coeff in pauli_terms:

            label = pauli.to_label()

            theta = params[p]

            active = []

            # ------------------------------------------------
            # Basis transformation
            # ------------------------------------------------

            for q, op in enumerate(label[::-1]):

                if op == 'X':

                    qc.h(q)
                    active.append(q)

                elif op == 'Y':

                    qc.sdg(q)
                    qc.h(q)

                    active.append(q)

                elif op == 'Z':

                    active.append(q)

            # ------------------------------------------------
            # Entangling chain
            # ------------------------------------------------

            for i in range(len(active) - 1):

                qc.cx(
                    active[i],
                    active[i + 1]
                )

            # ------------------------------------------------
            # exp(-i θ P)
            # ------------------------------------------------

            if len(active) > 0:

                qc.rz(
                    2 * np.real(coeff) * theta,
                    active[-1]
                )

            # ------------------------------------------------
            # Undo entangling chain
            # ------------------------------------------------

            for i in reversed(range(len(active) - 1)):

                qc.cx(
                    active[i],
                    active[i + 1]
                )

            # ------------------------------------------------
            # Undo basis transformation
            # ------------------------------------------------

            for q, op in enumerate(label[::-1]):

                if op == 'X':

                    qc.h(q)

                elif op == 'Y':

                    qc.h(q)
                    qc.s(q)

            p += 1

    return qc


# ============================================================
# Ansatz
# ============================================================

nqubits = 4


# HVA usually needs MUCH fewer layers
layers = 2

ansatz = hva_circuit(
    H_pauli_shifted,
    nqubits,
    layers
)

# ============================================================
# Estimator
# ============================================================

estimator = StatevectorEstimator()

# ============================================================
# Optimizer
# ============================================================

optimizer = SPSA(
    maxiter=500
)

# ============================================================
# VQE
# ============================================================

vqe = VQE(
    estimator=estimator,
    ansatz=ansatz,
    optimizer=optimizer
)

# ============================================================
# Run VQE
# ============================================================

print("\nRunning HVA-VQE...\n")

result = vqe.compute_minimum_eigenvalue(
    H_pauli_shifted
)

# ============================================================
# Shifted energy
# ============================================================

E_shifted = result.eigenvalue.real

# ============================================================
# Physical energy
# ============================================================

E_vqe = E_shifted + identity_shift

# ============================================================
# Optimal parameters
# ============================================================

optimal_params = result.optimal_parameters

# ============================================================
# Optimized circuit
# ============================================================

optimal_circuit = ansatz.assign_parameters(
    optimal_params
)

# ============================================================
# VQE state
# ============================================================

psi_vqe = Statevector.from_instruction(
    optimal_circuit
).data

# ============================================================
# Fidelity
# ============================================================

Fidelity = np.abs(
    np.vdot(psi_loc, psi_vqe)
)**2

# ============================================================
# Results
# ============================================================

print("\n================================")
print("Exact energy :", E0_loc)
print("VQE energy   :", E_vqe)
print("Energy error :", abs(E_vqe - E0_loc))
print("Fidelity     :", Fidelity)
print("Layers       :", layers)
print("Pauli terms  :", len(H_pauli_shifted))
print("================================")


NameError: name 'H_pauli_shifted' is not defined